# MSN PCT — 从零训练 SkullFix 颅骨补全

取代 `MSN_model_training_Demo.ipynb`。模型拓扑没变（LBR / offset self-attention /
cross-attention 解码器 / copy-and-mapping 上采样），改的是**能不能训得动**。

---

## ⚠️ 运行须知（先看这段）

**1. kernel 必须选 `comp0190-msn`**（右上角 Select Kernel）。这个环境有 TensorFlow 2.15；
另一个环境 `comp0190` 是 PyTorch 的，没有 TF。

**2. 单元格必须按顺序跑，尤其是"训练"要在"建模型"之前。**
这个 kernel 一旦建了模型就会占住显存，而训练子进程需要 15.5 GiB / 24 GiB。
顺序反了会 OOM。如果你已经建过模型又想训练，**先 Restart Kernel**。

**3. 数据准备已经做过了**，`data/cache/` 里的缓存可以直接用（下面第 1 节会自检）。
只有当你要改点数或样本量时才需要重跑，而且它**必须在另一个环境**里跑
（需要 `scikit-image`，只装在 `comp0190`）：

```bash
/root/miniconda3/envs/comp0190/bin/python src/data/prepare_skullfix.py \
    --n-samples 50 --n-dense 16384 --n-in 4096 --n-out 6144 --workers 12 \
    --out data/cache/skullfix_pairs_4096_6144.npz
```

**4. 训练想跑得更稳的话，用终端而不是 notebook**（断连不会中断）：

```bash
cd /root/comp0190-organ-completion
nohup /root/miniconda3/envs/comp0190-msn/bin/python src/models/train_skullfix.py \
    --minutes 55 > train.log 2>&1 &
tail -f train.log
```

notebook 里的训练单元格做的是同一件事，只是输出直接打在单元格里，关掉页面就没了。

---

## 相对 demo 改了什么

**A. 数据配对错位（正确性 bug）**
`explore_skull.ipynb` 的 `nrrd_to_point_cloud` 对 complete 和 defective **各自独立**做
`normalize_point_cloud`。切掉一块骨头会移动质心、改变最大半径，配对的两朵云于是落到
**不同坐标系**。实测 skull_000：质心偏移 7.55 voxel（半径的 3.6%）、尺度差 2.8%，
同点数下 GT→input 最近距离被抬高 32%。现在从 defective 推出**唯一一个**相似变换同时作用于两者
（推理时只有 defective 可用，所以这个坐标系可复现）。

**B. 距离计算把显存吃爆**
demo 的 `distance_matrix` 把两朵云 tile 成 `(B,N,M,3)`。按 demo 自己的设置（batch 8、6144 点）
光这一个张量前向就 ~12 GB，反向还要留着——这就是推理 notebook 当初被迫退回 CPU 的原因。
换成 `|a|²-2a·b+|b|²` 只产生 `(B,N,M)`。**改完之后完整论文架构（187.5M 参数）在单张 4090 上
372 ms/步、15.5 GiB，训得动了**，不需要缩模型。

**C. DCD 无法从随机初始化引导**
DCD 有界于 [0,2]，两个因子在预测偏离时同时消失。本模型初始化实测：最近邻距离均值 2.79 →
`exp(-2.79)=0.067`；6144 个 GT 点全部坍缩到 **6 个**不同预测点 → 密度权重 ~1/1970。
两者相乘让 loss 钉在 1.9995（上界 2.0），在 1e-7/1e-4/3e-4/1e-3 上扫 40 步 loss 变化不到 0.03。
默认改用 `cd_dcd`：CD 无界负责把形状拉对，DCD 在形状对上后接管精修。

**D. 推理不确定性**
demo 的 `UniformSampler` 用有状态随机数抽质心，**同一个模型对同一输入两次调用输出差 1.03**。
现在训练时保持随机（对 40 个样本相当于免费增广），推理时改用固定种子的 stateless 抽样，
重复调用逐位一致——指标才可复现。

**E. 其它**
- 学习率 1e-7 → 3e-4 + 100 步 warmup。1e-7 配 Adam 比常规小三个数量级。
- batch 8 → 4（24 GB 上 8 会 OOM）。模型里**没有 BatchNorm**（`LBR` 只是 Dense+ReLU），小 batch 只增加梯度噪声。
- `validation_split=0.1` → 按颅骨 id 显式划分。Keras 是**先切尾部再打乱**，一旦每颗颅骨生成多个 partial
  （demo 的 `preprocess_data` 就生成 2 个），兄弟样本会横跨划分边界造成泄漏。
- 冻结的 BERT 预计算。`trainable=False` 且只有 "skull" 一个类别 → 输出是常量，每步重算 1.1 亿参数是浪费。
- checkpoint 存成 `best.h5` 而非 `best.weights.h5`。后者是新版 Keras 格式，即使
  `save_weights_only=True` 也会把 Adam 的动量一起存（187.5M → 562M 个值，**2.25 GB**），
  且每次 val 改善都重写一遍。旧格式只存权重（750 MB）。
- 体素间距。nrrd 头里是各向异性带剪切的 `space directions`（0.451/0.446/0.625 mm），
  demo 在索引空间做 marching cubes，颅骨沿 z 被拉伸约 39%。现在应用该变换并存下 `scale_mm`，
  指标可直接换算成毫米。

## 1. 数据自检

只用 numpy，不碰 GPU。

In [1]:
import os, sys, json, subprocess
import numpy as np

REPO = os.path.abspath("../..")
CACHE = os.path.join(REPO, "data", "cache", "skullfix_pairs_4096_6144.npz")
RUN   = os.path.join(REPO, "experiments", "msn_skullfix")
PY_MSN = "/root/miniconda3/envs/comp0190-msn/bin/python"

assert os.path.exists(CACHE), f"Cache not found. Run prepare_skullfix.py first (see note 3):\n{CACHE}"

data = np.load(CACHE)
ids, inputs, gt = data["ids"], data["inputs"], data["gt"]
scale_mm = float(data["scale_mm"].mean())
print(f"{len(ids)} skull pairs | input {inputs.shape} | gt {gt.shape}")
print(f"Normalisation scale {scale_mm:.1f} mm  (normalised CD x scale_mm = mm)")


def nn_dist(query, ref, chunk=1024):
    """Nearest-neighbour distance from each query point to `ref`. Chunked pure
    numpy -- this env deliberately has no scipy: its versions are pinned tight
    (numpy<2 + tensorflow<2.16) and a diagnostic is not worth touching them."""
    ref2 = (ref ** 2).sum(1)
    out = np.empty(len(query), dtype=np.float64)
    for i in range(0, len(query), chunk):
        q = query[i:i + chunk]
        d2 = (q ** 2).sum(1)[:, None] - 2.0 * (q @ ref.T) + ref2[None, :]
        out[i:i + chunk] = np.sqrt(np.maximum(d2.min(1), 0.0))
    return out


# Pair-alignment self-check: distance from each GT point to the nearest input
# point. The median should sit at roughly the sampling spacing (shared surface)
# and only the tail (the defect) should be large. A large median means the two
# clouds are NOT in the same frame.
nn = nn_dist(gt[0].astype(np.float64), inputs[0].astype(np.float64))
print(f"\nskull_{ids[0]}  GT -> input nearest-neighbour distance:")
print(f"  median {np.median(nn)*scale_mm:6.2f} mm   <- shared surface")
print(f"  p99    {np.percentile(nn,99)*scale_mm:6.2f} mm   <- defect region")
print(f"  fraction beyond 0.05: {(nn>0.05).mean()*100:.1f}%")

50 skull pairs | input (50, 4096, 3) | gt (50, 6144, 3)
Normalisation scale 103.2 mm  (normalised CD x scale_mm = mm)

skull_000  GT -> input nearest-neighbour distance:
  median   2.73 mm   <- shared surface
  p99     20.26 mm   <- defect region
  fraction beyond 0.05: 5.3%


## 2. 训练

**这一格要在建模型之前跑**（本 kernel 还没占显存）。

`--minutes` 到点就停并落盘，所以任何预算都能拿到可用 checkpoint。单张 4090 上约 **4.7 s/epoch**
（40 训练 + 10 验证）。

### 该跑多久？实测的收敛曲线

| 预算 | epoch | val CD_t | val DCD | 说明 |
|---:|---:|---:|---:|---|
| 1 min | ~13 | ~19 mm | 1.65 | 只能看出形状在成型 |
| **3 min** | **~40** | **7.9 mm** | **1.00** | **推荐的首次跑通**，已经是像样的颅骨 |
| 5 min | ~60 | 7.6 mm | 0.93 | 再挤 4% |
| 7 min | ~87 | 7.1 mm | 0.88 | 再挤 6% |

**前 20 个 epoch 拿走 95% 的进步**（104.7 → 8.4 mm），之后收益断崖。先用 3 分钟跑通看结果，
确认管线和可视化都对了，再决定要不要拉长。

拉长到几十分钟只在"想榨干这批数据"时才有意义 —— 而 187.5M 参数对 40 颗颅骨主要是在记忆化，
所以榨出来的数字本身价值有限。真正的提升要靠加数据（`--n-samples 0` 上全部 100 对），不是加时间。

常用参数：`--config small` 换 9.4M 调试版（25 ms/步）· `--loss cd` 只用 Chamfer · `--lr` · `--batch-size`。

> **别拿这一轮的验证数字当结论。** train/val 有 gap 是预期的；一个**连 40 个样本都过拟合不了**的管线才是坏的。

In [2]:
MINUTES = 3      # 3 for a first pass; 5-10 to squeeze a bit more; 55 to exhaust this data

proc = subprocess.Popen(
    [PY_MSN, "src/models/train_skullfix.py", "--minutes", str(MINUTES)],
    cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
try:
    for line in proc.stdout:
        print(line, end="")
finally:
    proc.wait()
print("\nreturn code:", proc.returncode)

2026-07-28 04:41:43.993876: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-07-28 04:41:43.993896: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-07-28 04:41:43.994935: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
text feature (768,) (constant: frozen BERT, single class)

config=paper  params=187.5M  in=4096 out=6144
train=40 skulls  val=10 skulls (ids 013, 039, 030, 045, 017...)
loss=cd_dcd  lr=0.0003  batch=4  budget=3 min  scale=103.2 mm

Epoch 1/10000
I0000 00:00:1785213712.111100  165169 device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most o

## 3. 训练曲线

In [3]:
import pandas as pd
hist = pd.read_csv(os.path.join(RUN, "history.csv"))
meta = json.load(open(os.path.join(RUN, "run.json")))
print(f"epochs run {meta['epochs_run']} | best val_loss {meta['best_val_loss']:.4f} "
      f"| best val CD_t {meta['best_val_cd_t_mm']:.2f} mm")
print(f"validation skulls: {', '.join(meta['val_ids'])}")
hist.tail(3)

epochs run 40 | best val_loss 1.0468 | best val CD_t 8.12 mm
validation skulls: 013, 039, 030, 045, 017, 048, 026, 025, 032, 019


,epoch,cd_p_metric,cd_t_metric,dcd_metric,loss,lr,val_cd_p_metric,val_cd_t_metric,val_dcd_metric,val_loss
37,37,0.205004,0.084539,1.005924,1.090462,0.0003,0.19808,0.078722,0.962819,1.046803
38,38,0.206481,0.085587,1.018496,1.104083,0.0003,0.20452,0.083766,1.000753,1.088670
39,39,0.200775,0.080932,0.978318,1.059250,0.0003,0.19928,0.079624,0.972338,1.051854


In [4]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(rows=1, cols=3, subplot_titles=("Loss (CD + DCD)", "CD_t (mm)", "DCD"))
for col, (tr, va, mul) in enumerate([("loss", "val_loss", 1),
                                     ("cd_t_metric", "val_cd_t_metric", scale_mm),
                                     ("dcd_metric", "val_dcd_metric", 1)], start=1):
    fig.add_trace(go.Scatter(y=hist[tr] * mul, name="train", line=dict(color="#1565C0"),
                             showlegend=(col == 1)), row=1, col=col)
    fig.add_trace(go.Scatter(y=hist[va] * mul, name="validation", line=dict(color="#E64A19"),
                             showlegend=(col == 1)), row=1, col=col)
fig.update_xaxes(title_text="epoch")
fig.update_layout(height=380, legend=dict(itemsizing="constant"),
                  title="Training curves - the train/validation gap is memorisation")
fig.show()

## 4. 载入模型并评估

这里开始才占显存。指标定义跟 demo 一致（`distance_matrix` 返回欧氏距离而非平方距离），
所以 CD_p / CD_t 可以和 `skullfix_eval_results.csv` 对齐着看。

In [5]:
os.environ["HF_HOME"] = "/root/.cache/huggingface"
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
import tensorflow as tf
for _g in tf.config.experimental.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(_g, True)
sys.path.insert(0, os.path.join(REPO, "src", "models"))
import msn_skullfix as msn

cfg = msn.MSNConfig.paper()
model = msn.build_model(cfg)
model.load_weights(os.path.join(RUN, "best.h5"))
text_feat = np.load(os.path.join(REPO, "data", "cache", "bert_skull.npy"))
print(f"params={model.count_params()/1e6:.1f}M  in={cfg.n_in}  out={cfg.n_out}")

2026-07-28 04:44:48.561299: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-07-28 04:44:48.561316: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-07-28 04:44:48.562391: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


params=187.5M  in=4096  out=6144


In [6]:
val_ids = meta["val_ids"]
val_pos = [int(np.where(ids == v)[0][0]) for v in val_ids]

rows = []
for pos, sid in zip(val_pos, val_ids):
    pred = model([inputs[pos][None], text_feat[None]], training=False)
    g = tf.constant(gt[pos][None])
    cd_p, cd_t = msn.calc_cd(pred, g)
    rows.append({"id": sid, "CD_p": float(cd_p[0]), "CD_t": float(cd_t[0]),
                 "CD_t_mm": float(cd_t[0]) * scale_mm, "DCD": float(msn.calc_dcd(pred, g))})

val_df = pd.DataFrame(rows)
print(val_df[["CD_p", "CD_t", "CD_t_mm", "DCD"]].describe().loc[["mean", "std", "min", "max"]].round(4))
val_df

        CD_p    CD_t  CD_t_mm     DCD
mean  0.1984  0.0790   8.1476  0.9660
std   0.0102  0.0083   0.8560  0.0705
min   0.1865  0.0696   7.1773  0.8826
max   0.2190  0.0960   9.9005  1.0943


,id,CD_p,CD_t,CD_t_mm,DCD
0,013,0.197664,0.078198,8.066197,0.954597
1,039,0.196387,0.077153,7.958413,0.980067
2,030,0.191080,0.073026,7.532738,0.919674
3,045,0.218989,0.095980,9.900459,1.066136
4,017,0.191818,0.073591,7.590965,0.943739
5,048,0.212259,0.090116,9.295610,1.094346
6,026,0.191306,0.073232,7.553969,0.882551
7,025,0.195730,0.076630,7.904463,0.939862
8,032,0.186521,0.069580,7.177312,0.883949
9,019,0.202707,0.082360,8.495581,0.995145


## 5. 可视化

蓝色是预测的完整颅骨，浅色小点是缺损输入。缺损区应该只有蓝色。

In [7]:
PRED_COLOR    = "#1565C0"   # blue   - predicted complete skull
INPUT_COLOR   = "#EF9A9A"   # light red, drawn semi-transparent - defective input
GT_COLOR      = "#2E7D32"   # green  - ground truth


def show_completion(pos, sid, camera=(1.6, 1.6, 1.2)):
    pred = np.asarray(model([inputs[pos][None], text_feat[None]], training=False))[0]
    fig = go.Figure([
        go.Scatter3d(x=pred[:, 0], y=pred[:, 1], z=pred[:, 2], mode="markers",
                     name="Predicted complete skull",
                     marker=dict(size=1.6, color=PRED_COLOR, opacity=0.85)),
        go.Scatter3d(x=inputs[pos][:, 0], y=inputs[pos][:, 1], z=inputs[pos][:, 2],
                     mode="markers", name="Defective input",
                     marker=dict(size=1.5, color=INPUT_COLOR, opacity=0.45)),
    ])
    fig.update_layout(
        title=f"skull_{sid} (validation)  -  defect region should show blue only",
        height=650,
        # itemsizing="constant" -> legend swatches ignore the tiny marker size
        legend=dict(itemsizing="constant", x=0.02, y=0.98,
                    bgcolor="rgba(255,255,255,0.6)"),
        scene=dict(aspectmode="data",
                   xaxis_title="x", yaxis_title="y", zaxis_title="z",
                   camera=dict(eye=dict(x=camera[0], y=camera[1], z=camera[2]))),
        margin=dict(l=0, r=0, b=0, t=40),
    )
    fig.show()


best = val_df["CD_t"].idxmin()
show_completion(val_pos[best], val_df.loc[best, "id"])

In [8]:
def show_pred_vs_gt(pos, sid):
    pred = np.asarray(model([inputs[pos][None], text_feat[None]], training=False))[0]
    g = gt[pos]
    fig = make_subplots(rows=1, cols=2,
                        specs=[[{"type": "scatter3d"}, {"type": "scatter3d"}]],
                        subplot_titles=("Prediction", "Ground truth (complete)"))
    fig.add_trace(go.Scatter3d(x=pred[:, 0], y=pred[:, 1], z=pred[:, 2], mode="markers",
                               name="Predicted complete skull",
                               marker=dict(size=1.4, color=PRED_COLOR)), 1, 1)
    fig.add_trace(go.Scatter3d(x=g[:, 0], y=g[:, 1], z=g[:, 2], mode="markers",
                               name="Ground truth",
                               marker=dict(size=1.4, color=GT_COLOR)), 1, 2)
    fig.update_layout(height=520, title=f"skull_{sid}",
                      legend=dict(itemsizing="constant", orientation="h",
                                  x=0.5, xanchor="center", y=-0.02),
                      scene=dict(aspectmode="data"), scene2=dict(aspectmode="data"))
    fig.show()


show_pred_vs_gt(val_pos[best], val_df.loc[best, "id"])

## 6. 对照：作者发布的预训练权重

`skullfix_eval_results.csv` 是用 `MSN_weights3.h5` 在**旧的、配对错位的** .ply 上跑出来的，
只能当粗略参照，**不是干净的对照组**。要做严格对比，需要用同一份配对归一化的数据重跑那份权重
（`MSNConfig.paper()` 与它权重兼容）。

In [9]:
base = pd.read_csv("skullfix_eval_results.csv")
cmp = pd.DataFrame({
    "Pretrained weights (old data, 50 samples)": base[["CD_p", "CD_t", "DCD"]].mean(),
    "This run, from scratch (validation)":       val_df[["CD_p", "CD_t", "DCD"]].mean(),
}).round(4)
cmp

,"Pretrained weights (old data, 50 samples)","This run, from scratch (validation)"
CD_p,0.2203,0.1984
CD_t,0.0976,0.0790
DCD,1.4315,0.9660


## 下一步

1. **样本量**。50 → 全部 100 对（`--n-samples 0`）。真正的结论需要 SkullBreak 级别的量。
2. **跟预训练权重做干净对照**。用同一份配对数据重跑 `MSN_weights3.h5`，才能说清"从零训练 vs 微调"。
3. **数据增强**。40 样本 + 187M 参数，绕 SI 轴小角度旋转和轻微 jitter 值得试；
   但颅骨在 LPS 下方向一致，大角度随机旋转会破坏这个先验。
4. **体素基线**。这是项目的对比对象，评估指标（`src/eval/`）应在两条路线间共用。